In [1]:
import pandas as pd

raw = "../data/raw/"
machines  = pd.read_csv(raw + "PdM_machines.csv")
telemetry = pd.read_csv(raw + "PdM_telemetry.csv", parse_dates=["datetime"])
errors    = pd.read_csv(raw + "PdM_errors.csv", parse_dates=["datetime"])
maint     = pd.read_csv(raw + "PdM_maint.csv", parse_dates=["datetime"])
failures  = pd.read_csv(raw + "PdM_failures.csv", parse_dates=["datetime"])

print(machines.shape, telemetry.shape, errors.shape, maint.shape, failures.shape)

(100, 3) (876100, 6) (3919, 3) (3286, 3) (761, 3)


In [2]:
tables = {"machines": machines, "telemetry": telemetry, "errors": errors,
          "maint": maint, "failures": failures}

for name, df in tables.items():
    print("=====", name, "=====")
    print("Columns:", df.columns.tolist())
    print("Missing values:", df.isna().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())
    print(df.head(3))
    print()

===== machines =====
Columns: ['machineID', 'model', 'age']
Missing values: 0
Duplicate rows: 0
   machineID   model  age
0          1  model3   18
1          2  model4    7
2          3  model3    8

===== telemetry =====
Columns: ['datetime', 'machineID', 'volt', 'rotate', 'pressure', 'vibration']
Missing values: 0
Duplicate rows: 0
             datetime  machineID        volt      rotate    pressure  \
0 2015-01-01 06:00:00          1  176.217853  418.504078  113.077935   
1 2015-01-01 07:00:00          1  162.879223  402.747490   95.460525   
2 2015-01-01 08:00:00          1  170.989902  527.349825   75.237905   

   vibration  
0  45.087686  
1  43.413973  
2  34.178847  

===== errors =====
Columns: ['datetime', 'machineID', 'errorID']
Missing values: 0
Duplicate rows: 0
             datetime  machineID errorID
0 2015-01-03 07:00:00          1  error1
1 2015-01-03 20:00:00          1  error3
2 2015-01-04 06:00:00          1  error5

===== maint =====
Columns: ['datetime', 'machin

In [3]:
summary = pd.DataFrame({
    "rows": {n: len(d) for n, d in tables.items()},
    "missing_values": {n: int(d.isna().sum().sum()) for n, d in tables.items()},
    "duplicate_rows": {n: int(d.duplicated().sum()) for n, d in tables.items()},
})
print(summary)
print()
for n, d in tables.items():
    print(n, d.columns.tolist())


             rows  missing_values  duplicate_rows
machines      100               0               0
telemetry  876100               0               0
errors       3919               0               0
maint        3286               0               0
failures      761               0               0

machines ['machineID', 'model', 'age']
telemetry ['datetime', 'machineID', 'volt', 'rotate', 'pressure', 'vibration']
errors ['datetime', 'machineID', 'errorID']
maint ['datetime', 'machineID', 'comp']
failures ['datetime', 'machineID', 'failure']


## Data quality check
- 5 tables loaded: machines (100 rows), telemetry (876,100), errors (3,919), maint (3,286), failures (761)
- No missing values and no duplicate rows in any table
- All tables link through machineID

In [4]:
# Give the sensor columns clearer names
telemetry = telemetry.rename(columns={"volt": "voltage", "rotate": "rotation"})

# Check the range of each sensor (look for impossible values like negatives)
print(telemetry[["voltage", "rotation", "pressure", "vibration"]].describe().round(1))

# Check the date range and that every table uses the same 100 machines
print()
print("Telemetry dates:", telemetry["datetime"].min(), "to", telemetry["datetime"].max())
print("Machines in telemetry:", telemetry["machineID"].nunique())
print("Machines in failures:", failures["machineID"].nunique())

        voltage  rotation  pressure  vibration
count  876100.0  876100.0  876100.0   876100.0
mean      170.8     446.6     100.9       40.4
std        15.5      52.7      11.0        5.4
min        97.3     138.4      51.2       14.9
25%       160.3     412.3      93.5       36.8
50%       170.6     447.6     100.4       40.2
75%       181.0     482.2     107.6       43.8
max       255.1     695.0     186.0       76.8

Telemetry dates: 2015-01-01 06:00:00 to 2016-01-01 06:00:00
Machines in telemetry: 100
Machines in failures: 98


## Value checks
- All sensor readings are in plausible ranges (no negatives)
- Data covers 1 Jan 2015 to 1 Jan 2016 (one year, hourly)
- 100 machines in telemetry; 98 of them have at least one failure

In [5]:
# Operating hours
telemetry = telemetry.sort_values(["machineID", "datetime"]).reset_index(drop=True)
telemetry["hours_this_year"] = telemetry.groupby("machineID").cumcount()
telemetry = telemetry.merge(machines, on="machineID", how="left")
telemetry["operating_hours"] = telemetry["age"] * 8760 + telemetry["hours_this_year"]

print(telemetry[["machineID", "age", "hours_this_year", "operating_hours"]].head(3))

   machineID  age  hours_this_year  operating_hours
0          1   18                0           157680
1          1   18                1           157681
2          1   18                2           157682


In [6]:
import numpy as np
np.random.seed(42)   # makes the "random" numbers identical every run

# How far above or below average is each rotation speed?
z = (telemetry["rotation"] - telemetry["rotation"].mean()) / telemetry["rotation"].std()

# Base temperature: 70, a bit higher for faster rotation, plus random noise
telemetry["temperature"] = 70 + 3 * z + np.random.normal(0, 1.5, len(telemetry))

# Raise the temperature gradually in the 72 hours before each failure
fail_times = failures[["machineID", "datetime"]].drop_duplicates()
for _, f in fail_times.iterrows():
    start = f["datetime"] - pd.Timedelta(hours=72)
    mask = (telemetry["machineID"] == f["machineID"]) & \
           (telemetry["datetime"] >= start) & (telemetry["datetime"] <= f["datetime"])
    hours_to_fail = (f["datetime"] - telemetry.loc[mask, "datetime"]).dt.total_seconds() / 3600
    telemetry.loc[mask, "temperature"] += (72 - hours_to_fail) / 72 * 8

print(telemetry["temperature"].describe().round(1))

count    876100.0
mean         70.2
std           3.4
min          51.6
25%          68.0
50%          70.2
75%          72.5
max          89.7
Name: temperature, dtype: float64


## AI-generated columns
The Azure dataset has no temperature or operating-hours columns, so both were generated in Python:
- temperature: baseline 70 plus a term linked to rotation speed, random noise, and a rise of up to 8 degrees in the 72 hours before each failure
- operating_hours: machine age x 8,760 + hours elapsed in the telemetry year

In [7]:
# Label: does a failure happen within the next 24 hours?
telemetry["fail_within_24h"] = 0
for _, row in fail_times.iterrows():
    mask = (telemetry["machineID"] == row["machineID"]) & \
           (telemetry["datetime"] > row["datetime"] - pd.Timedelta(hours=24)) & \
           (telemetry["datetime"] <= row["datetime"])
    telemetry.loc[mask, "fail_within_24h"] = 1

print(telemetry["fail_within_24h"].value_counts())
print(telemetry[["machineID", "age", "operating_hours", "temperature", "fail_within_24h"]].head(3))

fail_within_24h
0    858899
1     17201
Name: count, dtype: int64
   machineID  age  operating_hours  temperature  fail_within_24h
0          1   18           157680    69.144598                0
1          1   18           157681    67.294727                0
2          1   18           157682    75.570284                0


In [8]:
telemetry.to_parquet("../data/processed/telemetry.parquet")
failures.to_parquet("../data/processed/failures.parquet")
maint.to_parquet("../data/processed/maint.parquet")
errors.to_parquet("../data/processed/errors.parquet")
machines.to_parquet("../data/processed/machines.parquet")
print("Saved")

Saved
